Install and Import packages

Allow Pandas dataframes to interface directly with a Postgres database engine

In [214]:
pip install sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


Safely read environment config variables from local files

In [215]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


Handles HTTP requests to pull raw data from the USAJobs API

In [216]:
import requests


Data Manipulation

In [217]:
import pandas as pd

In [218]:
import numpy as np

Engine creator used to establish the target SQL connection route

In [219]:
from sqlalchemy import create_engine

Handles dates and timestamps manipulation

In [220]:
from datetime import datetime

NLP tool to evaluate string similarities

In [221]:
from sklearn.feature_extraction.text import TfidfVectorizer

Tool for text cleaning patterns

In [222]:
import re

System library to read environment variables from the OS

In [223]:
import os

Imports the key-value secrets from a local .env file

In [224]:
from dotenv import load_dotenv

**Extract Layer**

In [225]:
roles = [
    "data analyst",
    "data scientist",
    "data engineer",
    "business analyst",
    "machine learning engineer",
    "analytics engineer",
    "data specialist",
    "research analyst",
    "business intelligence analyst",
    "data architect",
    "data warehouse engineer",
    "database administrator",
    
]

Loads hidden API keys and passwords from the .env file into Python's environment

In [226]:
load_dotenv()

True

Fetch my secret USAJobs API credentials securely from the system environment variables

In [227]:
USER_AGENT = os.getenv("USAJOBS_USER_AGENT")
AUTH_KEY = os.getenv("USAJOBS_AUTH_KEY")

Set up the API endpoint URL and the required security headers for authentication

In [228]:
url = "https://data.usajobs.gov/api/search"

headers = {
    "User-Agent": USER_AGENT,
    "Authorization-Key": AUTH_KEY
}

In [229]:
# Initialize an empty master list to store all collected job dictionaries
all_jobs = []
#Loop through my defined role list
for role in roles:

    page = 1 
    #Infinite loop to handle API pagination (continues until no more jobs are found)
    while True: 
     params = {
            "PositionTitle": role,
            "ResultsPerPage": 100,
            "Page": page,
            "Fields": "full",
          
        }
     # Send the GET request
     response = requests.get(url, headers=headers, params=params)
     data = response.json()
     # Extract the list of job postings. Return an empty list if none exist
     jobs = data.get("SearchResult", {}).get("SearchResultItems", [])
     # If the current page returns zero jobs, break the while loop and move to the next role
     if not jobs:
       break
     # Loop through each job
     for job in jobs:
            obj = job["MatchedObjectDescriptor"]
    
             # Get the is_remote data. As it's in the complex nested json format, I need to break it down
            remote_val = obj.get("RemoteIndicator")
            if str(remote_val).lower() == 'true':
                is_remote_val = True
            elif str(remote_val).lower() == 'false':
                is_remote_val = False
            else:
                is_remote_val = False 
        
            all_jobs.append({
                "id": job["MatchedObjectId"],
                "role": role,
                "title": obj.get("PositionTitle"),
                "location": obj.get("PositionLocationDisplay"),
                "organization": obj.get("OrganizationName"),
                "department": obj.get("DepartmentName"),
                "description": obj.get("UserArea", {}).get("Details", {}).get("JobSummary", ""),
                "posted_date": obj.get("PublicationStartDate"),
                "close_date": obj.get("ApplicationCloseDate"),
                "job_grade": obj.get("JobGrade"),
                "salary_min": obj.get("PositionRemuneration", [{}])[0].get("MinimumRange"),
                "salary_max": obj.get("PositionRemuneration", [{}])[0].get("MaximumRange"),
                "employment_type": obj.get("PositionSchedule", [{}])[0] if isinstance(obj.get("PositionSchedule"), list) else obj.get("PositionSchedule"),
                
                "is_remote": is_remote_val,  
            
                
            
            })
       
     # Fetch the next batch
     page += 1

 


In [230]:
df = pd.DataFrame(all_jobs)
print(df.shape)

(5588, 14)


In [231]:
df.head()

,id,role,title,location,organization,department,description,posted_date,close_date,job_grade,salary_min,salary_max,employment_type,is_remote
0,872719300,data analyst,Data Analyst,"Arlington, Virginia","Office of the Inspector General, USPS",Other Agencies and Independent Organizations,*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11T11:49:11.8070,2026-06-22T23:59:59.9970,[{'Code': 'GG'}],70623,158322,"{'Name': '', 'Code': '1'}",False
1,872439000,data analyst,Program Analyst (Data Analyst),Multiple Locations,Office of Personnel Management,Other Agencies and Independent Organizations,As a Program Analyst (Data Analyst) at the GS-...,2026-06-10T00:00:00.0000,2026-06-16T23:59:59.9970,[{'Code': 'GS'}],63795,99404,"{'Name': '', 'Code': '1'}",False
2,759326100,data analyst,Data Engineer,"Washington, District of Columbia",Central Intelligence Agency,Other Agencies and Independent Organizations,DDI Data Engineers solve critical data challen...,2025-10-01T00:00:00.0000,2026-09-30T23:59:59.9970,[{'Code': 'GS'}],74584,156755,"{'Name': '', 'Code': '1'}",False
3,846709300,data analyst,DATA SCIENTIST,Multiple Locations,Air Force Civilian Career Training,Department of the Air Force,The PALACE Acquire Program offers you a perman...,2025-09-29T00:00:00.0000,2026-09-28T23:59:59.9970,[{'Code': 'GS'}],49960,99314,"{'Name': '', 'Code': '1'}",False
4,856629100,data analyst,Data Scientist,"Woodlawn, Maryland",Social Security Administration,Other Agencies and Independent Organizations,IT Specialist (Data Scientist) positions are b...,2026-02-06T00:00:00.0000,2026-08-05T23:59:59.9970,[{'Code': 'GS'}],143913,187093,{'Name': 'Flextime and Alternate Work Schedule...,False


In [232]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5588 entries, 0 to 5587
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               5588 non-null   object
 1   role             5588 non-null   object
 2   title            5588 non-null   object
 3   location         5588 non-null   object
 4   organization     5588 non-null   object
 5   department       5588 non-null   object
 6   description      5588 non-null   object
 7   posted_date      5588 non-null   object
 8   close_date       5588 non-null   object
 9   job_grade        5588 non-null   object
 10  salary_min       5588 non-null   object
 11  salary_max       5588 non-null   object
 12  employment_type  5588 non-null   object
 13  is_remote        5588 non-null   bool  
dtypes: bool(1), object(13)
memory usage: 573.1+ KB


Check Data Types

In [233]:
df.dtypes

id                 object
role               object
title              object
location           object
organization       object
department         object
description        object
posted_date        object
close_date         object
job_grade          object
salary_min         object
salary_max         object
employment_type    object
is_remote            bool
dtype: object

Null Values

In [236]:
df.isnull().sum()/len(df)

id                 0.0
role               0.0
title              0.0
location           0.0
organization       0.0
department         0.0
description        0.0
posted_date        0.0
close_date         0.0
job_grade          0.0
salary_min         0.0
salary_max         0.0
employment_type    0.0
is_remote          0.0
dtype: float64

No Null Values

**Transform Layer**

Drop Duplicates

In [234]:
df = df.drop_duplicates(subset=["id"])

In [235]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2839 entries, 0 to 5220
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               2839 non-null   object
 1   role             2839 non-null   object
 2   title            2839 non-null   object
 3   location         2839 non-null   object
 4   organization     2839 non-null   object
 5   department       2839 non-null   object
 6   description      2839 non-null   object
 7   posted_date      2839 non-null   object
 8   close_date       2839 non-null   object
 9   job_grade        2839 non-null   object
 10  salary_min       2839 non-null   object
 11  salary_max       2839 non-null   object
 12  employment_type  2839 non-null   object
 13  is_remote        2839 non-null   bool  
dtypes: bool(1), object(13)
memory usage: 313.3+ KB


Drop irrelevant roles

Function to classify roles

In [237]:
def classify_role(title):

    t = str(title).lower()
    exclusions = ['operator', 'mechanic', 'clerk', 'laundry', 'bindery', 'vending', 'driver', 'automotive']
    if any(word in t for word in exclusions):
        return 'Other'
    if 'machine learning' in t or ' ml ' in t or ' ai ' in t or 'artificial intelligence' in t:
        return 'machine learning engineer'
        
    elif 'warehouse' in t or 'etl' in t:
        return 'data warehouse engineer'
        
    elif 'architect' in t or 'data modeling' in t:
        return 'data architect'
        
    elif 'analytics engineer' in t:
        return 'analytics engineer'
        
    elif 'intelligence' in t or ' bi ' in t:
        return 'business intelligence analyst'
        
    elif 'scientist' in t or 'data science' in t:
        return 'data scientist'
        
    elif 'engineer' in t:
        return 'data engineer'
        
    elif 'administrator' in t or ' dba ' in t or 'database' in t:
        return 'database administrator'
        
    elif 'specialist' in t and ('data' in t or 'info' in t):
        return 'data specialist'
        
    elif 'research' in t and ('analyst' in t or 'operations' in t):
        return 'research analyst'
        
    elif 'business' in t and 'analyst' in t:
        return 'business analyst'
        
    elif 'analyst' in t or 'analytics' in t or 'data' in t:
        return 'data analyst'
        
    else:
        return 'Other'


In [238]:
df['role'] = df['title'].apply(classify_role)

In [239]:
df=df[df['role'] != 'Other']

Transform Salary_min and Salary_max

Since salary_min and salary_max are decimal numbers, I'll change their data types

In [240]:
df['salary_min'] = pd.to_numeric(df['salary_min'], errors='coerce')
df['salary_max'] = pd.to_numeric(df['salary_max'], errors='coerce')

Since there are some jobs that are paid hourly, I need to normalize them to salary

In [241]:
df['salary_min'] = np.where(
    df['salary_min'] < 500, 
    df['salary_min'] * 2080, 
    df['salary_min']
)

In [242]:
df['salary_max'] = np.where(
    df['salary_max'] < 500, 
    df['salary_max'] * 2080, 
    df['salary_max']
)

Change Date Format

In [243]:
df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce").dt.date
df["close_date"] = pd.to_datetime(df["close_date"], errors="coerce").dt.date

Transform Job Grade

In [244]:
df["job_grade"] = df["job_grade"].apply(
    lambda x: x[0]["Code"] if isinstance(x, list) and len(x) > 0 else x
)

Transform Employment Type

In [245]:
df["employment_type"] = df["employment_type"].apply(
    lambda x: x.get("Code") if isinstance(x, dict) else x
)

Map Employement Type

In [246]:
e_map = {"1": "Full-Time",
       "2": "Part-Time",
       "3": "Shift Work",
       "4": "Intermittent",
       "5": "Job Share",
       "6": "On-Call"}

In [247]:
df['employment_type']=df['employment_type'].map(e_map).fillna(df['employment_type'])

Transform Location

In [248]:
us_states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", 
    "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", 
    "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", 
    "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", 
    "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio", 
    "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota", 
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", 
    "Wisconsin", "Wyoming", "District of Columbia"
]

In [249]:
# split location into city and state
df[["city","state"]]=df["location"].str.split(",",n=1,expand=True)
# remove spaces
df["city"]=df["city"].str.strip()
df["state"]=df["state"].str.strip()
# Fill in the rows that say "Multiple Locations"
df["state"]=df["state"].fillna("Multiple Locations")
df["city"]=df["city"].replace("Multiple Locations", "Various Cities")
#Filter out jobs that are not in the US
df=df[df["state"].isin(us_states)]

In [250]:
df.drop(columns="location")

,id,role,title,organization,department,description,posted_date,close_date,job_grade,salary_min,salary_max,employment_type,is_remote,city,state
0,872719300,data analyst,Data Analyst,"Office of the Inspector General, USPS",Other Agencies and Independent Organizations,*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11,2026-06-22,GG,70623.0,158322.0,Full-Time,False,Arlington,Virginia
2,759326100,data engineer,Data Engineer,Central Intelligence Agency,Other Agencies and Independent Organizations,DDI Data Engineers solve critical data challen...,2025-10-01,2026-09-30,GS,74584.0,156755.0,Full-Time,False,Washington,District of Columbia
4,856629100,data scientist,Data Scientist,Social Security Administration,Other Agencies and Independent Organizations,IT Specialist (Data Scientist) positions are b...,2026-02-06,2026-08-05,GS,143913.0,187093.0,Full-Time,False,Woodlawn,Maryland
8,871674500,data scientist,Data Scientist,U.S. Census Bureau,Department of Commerce,This vacancy is being filled via the Direct-Hi...,2026-06-04,2026-06-18,GG,102415.0,133142.0,Full-Time,False,Suitland,Maryland
10,872573900,data scientist,DATA SCIENTIST,Naval Air Systems Command,Department of the Navy,You will serve as a Data Scientist in the Anal...,2026-06-12,2026-06-16,GS,121785.0,158322.0,Full-Time,False,Patuxent River,Maryland
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5187,865386900,data warehouse engineer,Warehouse Worker,"Commander, Navy Installations Command",Department of the Navy,Support the Military's Quality of Life program...,2026-04-15,2026-06-30,NA,54100.8,63086.4,Full-Time,False,Everett,Washington
5203,872831600,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,U.S. Marine Corps,Department of the Navy,Marine Corps Community Services (MCCS) is look...,2026-06-11,2026-07-09,NF,33280.0,33280.0,Part-Time,False,Kaneohe,Hawaii
5213,871508100,data warehouse engineer,"Material Handler Supervisor, (Warehouse Worker...",Bureau of Prisons/Federal Prison System,Department of Justice,Corrections professionals who foster a humane ...,2026-06-03,2026-06-17,WS,59883.2,69888.0,Full-Time,False,Glenville,West Virginia
5216,871772900,data warehouse engineer,Materials Handler Supervisor (Warehouse Worker...,Bureau of Prisons/Federal Prison System,Department of Justice,Corrections professionals who foster a humane ...,2026-06-04,2026-06-26,WS,77105.6,89960.0,Full-Time,False,Marion,Illinois


In [251]:
order=["id","role", "title", "organization","department","city", "state", "description", "posted_date", "close_date", "job_grade", "salary_min", "salary_max", "employment_type", "is_remote"]

In [252]:
df=df[order]

In [253]:
df.dtypes

id                  object
role                object
title               object
organization        object
department          object
city                object
state               object
description         object
posted_date         object
close_date          object
job_grade           object
salary_min         float64
salary_max         float64
employment_type     object
is_remote             bool
dtype: object

In [254]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 816 entries, 0 to 5220
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               816 non-null    object 
 1   role             816 non-null    object 
 2   title            816 non-null    object 
 3   organization     816 non-null    object 
 4   department       816 non-null    object 
 5   city             816 non-null    object 
 6   state            816 non-null    object 
 7   description      816 non-null    object 
 8   posted_date      816 non-null    object 
 9   close_date       816 non-null    object 
 10  job_grade        816 non-null    object 
 11  salary_min       816 non-null    float64
 12  salary_max       816 non-null    float64
 13  employment_type  816 non-null    object 
 14  is_remote        816 non-null    bool   
dtypes: bool(1), float64(2), object(12)
memory usage: 96.4+ KB


Extract key skills

In [255]:
df_1=df.copy()

In [256]:
skill_filter = [
    "python", "sql", "tableau", "power bi", "excel",
    "aws", "spark", "snowflake", "machine learning"
]

In [257]:
texts = df_1["description"].fillna("").str.lower()

In [258]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
X = vectorizer.fit_transform(df_1["description"].fillna("").str.lower())

In [259]:
feature_names = vectorizer.get_feature_names_out()

In [260]:
# Get top keywords. I try to give those skill filter a boost
def get_top_keywords(row, top_n=10):
    row = row.toarray().flatten()

    scored_words = []

    for i, score in enumerate(row):
        word = feature_names[i]

        if score > 0:
            # base TF-IDF score
            final_score = score

            # BOOST if it's a skill
            if word in skill_filter:
                final_score *= 3

            scored_words.append((word, final_score))

    # sort by final score
    scored_words.sort(key=lambda x: x[1], reverse=True)

    return [word for word, score in scored_words[:top_n]]

In [261]:
df_1["top_keywords"] = [
    get_top_keywords(X[i]) for i in range(X.shape[0])
]

In [262]:
df_1

,id,role,title,organization,department,city,state,description,posted_date,close_date,job_grade,salary_min,salary_max,employment_type,is_remote,top_keywords
0,872719300,data analyst,Data Analyst,"Office of the Inspector General, USPS",Other Agencies and Independent Organizations,Arlington,Virginia,*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11,2026-06-22,GG,70623.0,158322.0,Full-Time,False,"[postal, united, states, service, note, inspec..."
2,759326100,data engineer,Data Engineer,Central Intelligence Agency,Other Agencies and Independent Organizations,Washington,District of Columbia,DDI Data Engineers solve critical data challen...,2025-10-01,2026-09-30,GS,74584.0,156755.0,Full-Time,False,"[data, building, analyzing, engineers, challen..."
4,856629100,data scientist,Data Scientist,Social Security Administration,Other Agencies and Independent Organizations,Woodlawn,Maryland,IT Specialist (Data Scientist) positions are b...,2026-02-06,2026-08-05,GS,143913.0,187093.0,Full-Time,False,"[civil, appointments, service, new, authority,..."
8,871674500,data scientist,Data Scientist,U.S. Census Bureau,Department of Commerce,Suitland,Maryland,This vacancy is being filled via the Direct-Hi...,2026-06-04,2026-06-18,GG,102415.0,133142.0,Full-Time,False,"[census, suitland, bureau, line, maryland, hea..."
10,872573900,data scientist,DATA SCIENTIST,Naval Air Systems Command,Department of the Navy,Patuxent River,Maryland,You will serve as a Data Scientist in the Anal...,2026-06-12,2026-06-16,GS,121785.0,158322.0,Full-Time,False,"[analytics, commander, river, maryland, fleet,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5187,865386900,data warehouse engineer,Warehouse Worker,"Commander, Navy Installations Command",Department of the Navy,Everett,Washington,Support the Military's Quality of Life program...,2026-04-15,2026-06-30,NA,54100.8,63086.4,Full-Time,False,"[recreation, cnic, navy, commander, installati..."
5203,872831600,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,U.S. Marine Corps,Department of the Navy,Kaneohe,Hawaii,Marine Corps Community Services (MCCS) is look...,2026-06-11,2026-07-09,NF,33280.0,33280.0,Part-Time,False,"[mccs, marine, corps, community, team, brighte..."
5213,871508100,data warehouse engineer,"Material Handler Supervisor, (Warehouse Worker...",Bureau of Prisons/Federal Prison System,Department of Justice,Glenville,West Virginia,Corrections professionals who foster a humane ...,2026-06-03,2026-06-17,WS,59883.2,69888.0,Full-Time,False,"[foster, humane, reentry, corrections, individ..."
5216,871772900,data warehouse engineer,Materials Handler Supervisor (Warehouse Worker...,Bureau of Prisons/Federal Prison System,Department of Justice,Marion,Illinois,Corrections professionals who foster a humane ...,2026-06-04,2026-06-26,WS,77105.6,89960.0,Full-Time,False,"[foster, humane, reentry, corrections, individ..."


=> This didn't work, as the keywords extracted were not important skills

Mapping method

In [263]:
'''
skills = [
    # programming
    "python", "r", "sql", "java",

    # data tools
    "excel", "tableau", "power bi",

    # cloud / engineering
    "aws", "azure", "snowflake", "spark", "databricks",

    # analytics / ml
    "data analysis", "statistics", "machine learning",
    "regression", "forecasting",

    # business
    "data visualization", "reporting", "etl"
]
'''


'\nskills = [\n    # programming\n    "python", "r", "sql", "java",\n\n    # data tools\n    "excel", "tableau", "power bi",\n\n    # cloud / engineering\n    "aws", "azure", "snowflake", "spark", "databricks",\n\n    # analytics / ml\n    "data analysis", "statistics", "machine learning",\n    "regression", "forecasting",\n\n    # business\n    "data visualization", "reporting", "etl"\n]\n'

In [264]:
'''
df["skills_found"] = df["description"].str.lower().apply(
    lambda text: [
        skill for skill in skills
        if re.search(rf"\b{re.escape(skill)}\b", text) # make the keyword matching more precise
    ]
)
'''


'\ndf["skills_found"] = df["description"].str.lower().apply(\n    lambda text: [\n        skill for skill in skills\n        if re.search(rf"\x08{re.escape(skill)}\x08", text) # make the keyword matching more precise\n    ]\n)\n'

In [265]:
'''
df["skills_found"] = df["skills_found"].apply(
    lambda x: x if isinstance(x, list) else []
)
'''

'\ndf["skills_found"] = df["skills_found"].apply(\n    lambda x: x if isinstance(x, list) else []\n)\n'

=> This didn't work as these skills may not appear in all jobs

**Load Layer**

In [266]:
df.to_csv("job_skills.csv", index=False)

In [267]:
df.columns = df.columns.str.lower().str.strip()
df = df.reset_index(drop=True)



In [268]:
df

,id,role,title,organization,department,city,state,description,posted_date,close_date,job_grade,salary_min,salary_max,employment_type,is_remote
0,872719300,data analyst,Data Analyst,"Office of the Inspector General, USPS",Other Agencies and Independent Organizations,Arlington,Virginia,*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11,2026-06-22,GG,70623.0,158322.0,Full-Time,False
1,759326100,data engineer,Data Engineer,Central Intelligence Agency,Other Agencies and Independent Organizations,Washington,District of Columbia,DDI Data Engineers solve critical data challen...,2025-10-01,2026-09-30,GS,74584.0,156755.0,Full-Time,False
2,856629100,data scientist,Data Scientist,Social Security Administration,Other Agencies and Independent Organizations,Woodlawn,Maryland,IT Specialist (Data Scientist) positions are b...,2026-02-06,2026-08-05,GS,143913.0,187093.0,Full-Time,False
3,871674500,data scientist,Data Scientist,U.S. Census Bureau,Department of Commerce,Suitland,Maryland,This vacancy is being filled via the Direct-Hi...,2026-06-04,2026-06-18,GG,102415.0,133142.0,Full-Time,False
4,872573900,data scientist,DATA SCIENTIST,Naval Air Systems Command,Department of the Navy,Patuxent River,Maryland,You will serve as a Data Scientist in the Anal...,2026-06-12,2026-06-16,GS,121785.0,158322.0,Full-Time,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
811,865386900,data warehouse engineer,Warehouse Worker,"Commander, Navy Installations Command",Department of the Navy,Everett,Washington,Support the Military's Quality of Life program...,2026-04-15,2026-06-30,NA,54100.8,63086.4,Full-Time,False
812,872831600,data warehouse engineer,SALES ASSOCIATE NF1 (RPT) MCX RETAIL WAREHOUSE,U.S. Marine Corps,Department of the Navy,Kaneohe,Hawaii,Marine Corps Community Services (MCCS) is look...,2026-06-11,2026-07-09,NF,33280.0,33280.0,Part-Time,False
813,871508100,data warehouse engineer,"Material Handler Supervisor, (Warehouse Worker...",Bureau of Prisons/Federal Prison System,Department of Justice,Glenville,West Virginia,Corrections professionals who foster a humane ...,2026-06-03,2026-06-17,WS,59883.2,69888.0,Full-Time,False
814,871772900,data warehouse engineer,Materials Handler Supervisor (Warehouse Worker...,Bureau of Prisons/Federal Prison System,Department of Justice,Marion,Illinois,Corrections professionals who foster a humane ...,2026-06-04,2026-06-26,WS,77105.6,89960.0,Full-Time,False


In [269]:
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DATABASE = os.getenv("DB_NAME")

In [270]:
database_url = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(database_url)

Check if the data already existed

In [271]:
try:
    df_existing = pd.read_sql("SELECT * FROM job_skills", engine)
except Exception:
    df_existing = pd.DataFrame()

In [272]:
df_combined = pd.concat([df_existing, df], ignore_index=True)


In [273]:
df_combined = df_combined.drop_duplicates(
    subset=["id"], 
    keep="last"
)

In [274]:
df_combined.to_sql("job_skills", engine, if_exists="replace", index=False)

816